In [ ]:
import os
os.chdir("..")

In [ ]:
from CensusForge import CensusAPI
from jp_imports import TradeUtils
import requests
import polars as pl
import numpy as np
import tempfile
from pathlib import Path
import duckdb
from jp_imports import download

tu = TradeUtils()
capi = CensusAPI()


In [ ]:
file_path = Path(f"data/raw/int_data.parquet")
if not file_path.exists() or update:
    download(
        url="http://apps.estadisticas.pr/iepr/LinkClick.aspx?fileticket=JVyYmIHqbqc%3d&tabid=284&mid=244930",
        filename=file_path,
    )

In [ ]:
file_path = Path(f"{self.saving_dir}raw/int_data.parquet")
if not file_path.exists() or update:
    download(
        url="http://apps.estadisticas.pr/iepr/LinkClick.aspx?fileticket=JVyYmIHqbqc%3d&tabid=284&mid=244930",
        filename=f"{tempfile.gettempdir()}/{hash(file_path)}.zip",
    )
    # Extract the zip file
    with zipfile.ZipFile(f"{tempfile.gettempdir()}/{hash(file_path)}.zip", "r") as zip_ref:
        zip_ref.extractall(f"{tempfile.gettempdir()}/{hash(file_path)}/")

    # Extract additional zip files
    additional_files = ["EXPORT_HTS10_ALL.zip", "IMPORT_HTS10_ALL.zip"]
    for additional_file in additional_files:
        additional_file_path = os.path.join(
            f"{tempfile.gettempdir()}/{hash(file_path)}/{additional_file}"
        )
        with zipfile.ZipFile(additional_file_path, "r") as zip_ref:
            zip_ref.extractall(
                os.path.join(f"{tempfile.gettempdir()}/{hash(file_path)}/")
            )

    # Concatenate the files
    imports = pl.scan_csv(
        f"{tempfile.gettempdir()}/{hash(file_path)}/IMPORT_HTS10_ALL.csv",
        ignore_errors=True,
    )
    exports = pl.scan_csv(
        f"{tempfile.gettempdir()}/{hash(file_path)}/EXPORT_HTS10_ALL.csv",
        ignore_errors=True,
    )
    pl.concat([imports, exports], how="vertical").collect().write_parquet(
        self.saving_dir + "raw/int_data.parquet"
    )

    logging.info(
        "finished extracting data from the Puerto Rico Institute of Statistics"
    )

In [ ]:
tu.pull_int_org()

In [ ]:
tu.pull_census_hts(exports=True, state="PR")

In [ ]:
df =pl.DataFrame(r)
df = df.rename(df.row(0, named=True))
df = df.slice(1)
df

In [ ]:
# fips = ["PR", "VI", "HI" ]
r = CensusAPI().timeseries_query(
                dataset="timeseries-intltrade-exports-statenaics",
                params_list=[
                    "CTY_CODE",
                    "CTY_NAME",
                    "ALL_VAL_MO",
                    "COMM_LVL",
                    "NAICS",
                ],
                year=2019,
                extra=f"STATE=HI",
                skip_checks=True,
            )